In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from gtf_utils import gtf_to_df_with_genes, save_gtf_subset
from model_utils import (
    get_high_confident_proteins,
    get_low_confident_proteins,
    find_apply_score_filter,
    train_iterative_model
)
from .plotting import (
    plot_roc_curve,
    plot_shap_summary,
    plot_stacked_percentage_bar,
    plot_sequence_coverage_groups_hist,
    plot_transcript_scores,
    plot_psauron_scores
)

ImportError: attempted relative import with no known parent package

In [ ]:
output_dir="/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/outputs/gapms/helixer/"
all_scores_path = "/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/outputs/gapms/helixer/all_proteins_scores.tsv"
all_scores_df = pd.read_csv(all_scores_path, sep='\t')

: 

In [ ]:
all_scores_df

,Protein,protein_length,protein_isoforms,splice_sites,prediction_score,sequence_coverage,mapped_peptides,N_terminal_peptides,C_terminal_peptides,protein_specific_peptides,gene_specific_peptides,splice_peptides,internal_peptides,psauron_score
0,GCF_000499845.2_NC_023752.2_000729.1,4755.0,1.0,49,0.55250,0.4749,175,0,0,171,171,18,157,0.99893
1,GCF_000499845.2_NC_023752.2_002637.1,5092.0,1.0,13,0.67813,0.3782,172,0,0,170,170,8,164,0.97442
2,GCF_000499845.2_NC_023758.2_002310.1,3750.0,1.0,13,0.83284,0.3771,111,0,1,105,105,6,105,0.98822
3,GCF_000499845.2_NC_023752.2_001939.1,3646.0,1.0,15,0.81370,0.3982,111,0,1,78,78,5,106,0.98303
4,GCF_000499845.2_NC_023751.2_002493.1,2260.0,1.0,30,0.67143,0.6128,110,0,0,110,110,20,90,0.99524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26368,GCF_000499845.2_NC_023756.2_000201.1,130.0,1.0,0,0.94310,0.0000,0,0,0,0,0,0,0,0.99996
26369,GCF_000499845.2_NC_023756.2_000202.1,976.0,1.0,0,0.72959,0.0000,0,0,0,0,0,0,0,0.95370
26370,GCF_000499845.2_NC_023756.2_000203.1,960.0,1.0,0,0.65052,0.0000,0,0,0,0,0,0,0,0.95573
26371,GCF_000499845.2_NC_023756.2_000208.1,196.0,1.0,0,0.97712,0.0000,0,0,0,0,0,0,0,0.99817


: 

In [ ]:
# Step 4: Find highly supported proteins
high_confident_df = get_high_confident_proteins(all_scores_df)
high_confident_proteins = set(high_confident_df['Protein'].unique())

: 

In [ ]:
print(len(all_scores_df))
print(len(all_scores_df[all_scores_df['mapped_peptides'] > 0]))
print(len(all_scores_df[all_scores_df["protein_specific_peptides"] >= 4]))
print(len(high_confident_proteins))
print(len(all_scores_df[all_scores_df["sequence_coverage"] >= 0.5]))

26373
16870
8798
9062
2030


: 

In [ ]:
# Step 5: Apply ROC-based score filtering
cutoff, score_supported = find_apply_score_filter(
    all_scores_df, high_confident_proteins,
    output_dir=output_dir,
    plot_roc=plot_roc_curve)

low_confident_df = get_low_confident_proteins(all_scores_df, cutoff)
low_confident_proteins = set(low_confident_df['Protein'].unique())

Determined cutoff: 1


: 

: 

In [ ]:
FEATURE_COLUMNS = [
"sequence_coverage",
"protein_specific_peptides",
"gene_specific_peptides",
"splice_peptides",
"internal_peptides",
"N_terminal_peptides",
"C_terminal_peptides",
"splice_sites",
"protein_isoforms"
]
# Step 6: Train iterative self-labeling model
labeled_df = train_iterative_model(
    all_scores_df,
    feature_cols=FEATURE_COLUMNS,
    pos_thr=0.90,
    neg_thr=0.10,
    n_iter=5,
    shap_output_dir=output_dir,
    plot_shap=plot_shap_summary
)

# Save iterative results
iterative_proteins = set(labeled_df[labeled_df["final_label"] == 1]["Protein"])
all_proteins = set(all_scores_df["Protein"])
supported_proteins = score_supported | iterative_proteins
unsupported_proteins = all_proteins - supported_proteins

: 

In [ ]:

    print(f"\nPipeline completed\nOutputs written to: {output_dir}")
    print(f"\nNumber of all proteins = {len(set(all_scores_df['Protein']))}")
    print(f"Number of high confident proteins = {len(high_confident_proteins)}")
    print(f"Number of low confident proteins = {len(low_confident_proteins)}")
    print(f"Number of prediction_only proteins = {len(score_supported) - len(high_confident_proteins)}")
    print(f"Number of iterative_only proteins = {len(iterative_proteins) - len(score_supported)}")
    print(f"Number of all supported proteins = {len(supported_proteins)}")
    print(f"Number of all un-supported proteins = {len(unsupported_proteins)}")

: 

: 

: 

: 

: 

: 